# Chicago Divvy ride duration

Do members and casual riders take trips of different lengths on weekdays
and weekends? This notebook compares ride duration across those groups using
anonymized Chicago-area trips published by Divvy for January 2021.
[Divvy system data](https://divvybikes.com/system-data)

Run the cells in order in a Python 3 kernel with pandas and Matplotlib installed.
The loading cell downloads the [January 2021 ZIP archive](https://divvy-tripdata.s3.amazonaws.com/202101-divvy-tripdata.zip)
directly from Divvy, so it requires internet access.

## Disclosure

I had Codex create this example document to do the assigment on. I did not have any notebooks that I thought would be good candidates for this assignment. I would have liked to use my Milestone 1 notebooks, but I already "producionalized" that code into separate files, scripts, and packages.

The work of converting this notebook apart into a script is all done on my own.

## Load and inspect the rides

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import matplotlib.pyplot as plt
import pandas as pd

archive_name = "202101-divvy-tripdata"
archive_url = f"https://divvy-tripdata.s3.amazonaws.com/{archive_name}.zip"
trip_columns = ["ride_id", "member_casual", "started_at", "ended_at"]

with urlopen(archive_url, timeout=30) as response:
    archive_bytes = response.read()

# Open the trip CSV by name because the ZIP also contains macOS metadata.
with ZipFile(BytesIO(archive_bytes)) as archive:
    with archive.open(f"{archive_name}.csv") as trips_file:
        raw_rides = pd.read_csv(trips_file, usecols=trip_columns)

print(f"Rows loaded: {len(raw_rides):,}")
raw_rides.head()

## Clean the trip records

Standardize rider labels, remove duplicate records, and parse the timestamps.
A trip needs a rider type and two valid timestamps, and its end time must be
after its start time. Station details are unnecessary for this comparison, so
trips with missing station information can still contribute.

In [ ]:
rides = raw_rides.rename(columns={"member_casual": "rider_type"})
rides["rider_type"] = rides["rider_type"].str.strip().str.lower().replace("", pd.NA)
rides = rides.drop_duplicates()

# Invalid timestamp strings become missing values so they can be filtered together.
rides["started_at"] = pd.to_datetime(rides["started_at"], errors="coerce")
rides["ended_at"] = pd.to_datetime(rides["ended_at"], errors="coerce")
rides = rides.dropna(subset=["rider_type", "started_at", "ended_at"]).copy()

rides["duration_minutes"] = (
    (rides["ended_at"] - rides["started_at"]).dt.total_seconds() / 60
)
rides = rides.loc[rides["duration_minutes"] > 0].copy()
print(f"Kept {len(rides):,} rides and removed {len(raw_rides) - len(rides):,} rows.")
rides.head()

## Summarize duration by rider type and day

Some recorded rides are unusually long. The summary includes the median because
very long rides can pull the mean upward.

In [ ]:
# Classify each trip using its start date. Pandas numbers Monday as 0.
rides["day_type"] = rides["started_at"].dt.dayofweek.ge(5).map(
    {False: "Weekday", True: "Weekend"}
)

summary = rides.groupby(["rider_type", "day_type"]).agg(
    ride_count=("ride_id", "size"),
    mean_minutes=("duration_minutes", "mean"),
    median_minutes=("duration_minutes", "median"),
)
summary.round(1)

## Compare the group means

In [ ]:
mean_by_day = summary["mean_minutes"].unstack("day_type")
ax = mean_by_day.plot.bar(
    figsize=(7, 4),
    rot=0,
    color=["#326D9A", "#DF923A"],
    title="Average Divvy ride duration, January 2021",
)
ax.set_xlabel("Rider type")
ax.set_ylabel("Mean duration (minutes)")
ax.legend(title="Trip start day")
plt.tight_layout()
plt.show()

## Predict ride duration with a simple model

Predict a ride's duration using the median duration of earlier trips with the
same rider type and day type. Both inputs are available when the ride starts.

Train on trips that finished before the final seven calendar days and test on
trips that started during those seven days. Use the overall training median
as a baseline and as a fallback for any rider and day combination absent from
training. Mean absolute error measures how far predictions are from the
observed durations, in minutes.

In [ ]:
test_start = rides["started_at"].max().normalize() - pd.Timedelta(days=6)
train_rides = rides.loc[rides["ended_at"] < test_start]
test_rides = rides.loc[rides["started_at"] >= test_start].copy()

model_features = ["rider_type", "day_type"]
baseline_minutes = train_rides["duration_minutes"].median()
group_medians = train_rides.groupby(model_features)["duration_minutes"].median()

test_rides = test_rides.join(
    group_medians.rename("predicted_minutes"), on=model_features
)
test_rides["predicted_minutes"] = test_rides["predicted_minutes"].fillna(
    baseline_minutes
)

actual_minutes = test_rides["duration_minutes"]
baseline_mae = (actual_minutes - baseline_minutes).abs().mean()
model_mae = (actual_minutes - test_rides["predicted_minutes"]).abs().mean()

print(f"Test trips starting on or after {test_start:%Y-%m-%d}: {len(test_rides):,}")
print(f"Baseline mean absolute error: {baseline_mae:.2f} minutes")
print(f"Model mean absolute error: {model_mae:.2f} minutes")